# DriveSense-VLM — 05: Quick Start (Full Pipeline)

**GPU**: A100 80GB | **Time**: ~8–10 h | **Cost**: ~110 CU

Runs the entire pipeline end-to-end. Edit the config block then run all cells.

| Stage | CU |
|-------|----|
| Data pipeline | ~5 |
| SFT training (~6 h) | ~72 |
| Optimization (~1.5 h) | ~18 |
| Benchmarks (~1 h) | ~12 |
| Evaluation (~1 h) | ~12 |
| **Total** | **~119** |

> ⚠️ **Before running**: Runtime → Change runtime type → **A100 GPU**
>
> **RECOVERY**: Rerun Cells 2–3, then rerun the interrupted stage. Every stage is idempotent.

In [ ]:
# ══════════════════════════════════════════════════════════
# CONFIGURATION — Edit these before running
# ══════════════════════════════════════════════════════════
FORCE_RERUN       = True   # True: always rerun each step (overwrite outputs)
USE_MOCK_LLM      = True   # True: free mock annotations; False: real Claude API (~$5-8)
INSTALL_FLASH_ATTN = False  # True: install flash-attn (15+ min compile); False: SDPA fallback
RUN_DEBUG_FIRST   = True   # True: run 10-step smoke test before full training
NUSCENES_VERSION  = "v1.0-mini"
MIN_RARITY_SCORE  = 1
# ══════════════════════════════════════════════════════════

# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD (skip if already cloned — saves bandwidth)
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)
print(f"Working dir: {os.getcwd()}")

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

In [ ]:
# Install ALL dependencies for the full pipeline
# ⚠️  NEVER install numpy or pandas — Colab's pre-installed versions are compiled
#     against numpy 2.x ABI. nuscenes-devkit installed with --no-deps.
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install nuscenes-devkit --no-deps -q 2>&1 | tail -1
!pip install pyquaternion matplotlib tqdm Pillow pyyaml requests openpyxl -q 2>&1 | tail -1
!pip install scikit-learn scipy pyspark -q 2>&1 | tail -1
!pip install transformers peft accelerate datasets bitsandbytes wandb -q 2>&1 | tail -1
!pip install autoawq -q 2>&1 | tail -1
!pip install tensorrt -q 2>&1 | tail -1 || echo "TensorRT not available — torch.compile fallback"

if INSTALL_FLASH_ATTN:
    print("Installing flash-attn (may take 15+ min)...")
    !pip install flash-attn --no-build-isolation -q 2>&1 | tail -3
else:
    print("Flash-attn: skipped (INSTALL_FLASH_ATTN=False). SDPA fallback will be used.")

import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → A100 GPU"
print(f"✓ GPU : {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import numpy as np, pandas as pd, drivesense
print(f"✓ numpy {np.__version__} | pandas {pd.__version__} | DriveSense loaded")

In [ ]:
import os, glob, wandb

TRAIN_OUT = f"{OUTPUTS_ROOT}/training"
os.makedirs(TRAIN_OUT, exist_ok=True)
ckpts = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
if ckpts:
    print(f"Found {len(ckpts)} existing checkpoint(s) — training will resume.")
    print(f"  Latest: {ckpts[-1]}")
else:
    print("No checkpoints — will start fresh training.")

wandb.login()
print("✓ W&B configured")

## Stage 1: Data Pipeline

`FORCE_RERUN=True` (default): clears and reruns each step.
`USE_MOCK_LLM=True` (default): free mock annotations.

In [ ]:
import os, json, shutil

os.chdir(REPO_ROOT)

# ── nuScenes rarity filtering ──────────────────────────────────────────────
FILTER_OUTPUT = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
NUSCENES_DIR  = f"{DATA_ROOT}/nuscenes"
json_path     = f"{FILTER_OUTPUT}/metadata.json"
jsonl_path    = f"{FILTER_OUTPUT}/metadata.jsonl"

output_exists = os.path.exists(json_path)
if output_exists and not FORCE_RERUN:
    with open(jsonl_path) as f:
        n = sum(1 for _ in f)
    print(f"✓ Filtering done ({n} frames). FORCE_RERUN=True to rebuild.")
else:
    if os.path.exists(FILTER_OUTPUT):
        shutil.rmtree(FILTER_OUTPUT)
    os.makedirs(FILTER_OUTPUT, exist_ok=True)
    !python scripts/run_nuscenes_filter.py \
        --nuscenes-root {NUSCENES_DIR} \
        --version {NUSCENES_VERSION} \
        --output-dir {FILTER_OUTPUT} \
        --min-score {MIN_RARITY_SCORE} \
        2>&1 | tail -5
    if os.path.exists(json_path) and not os.path.exists(jsonl_path):
        with open(json_path) as f:
            frames = json.load(f)
        with open(jsonl_path, "w") as f:
            for frame in frames:
                f.write(json.dumps(frame) + "\n")
        print(f"  ✓ Converted {len(frames)} frames → metadata.jsonl")

# ── PySpark ETL ────────────────────────────────────────────────────────────
SPARK_OUTPUT = f"{OUTPUTS_ROOT}/data/spark_processed"
spark_exists = os.path.exists(SPARK_OUTPUT) and bool(os.listdir(SPARK_OUTPUT)) if os.path.exists(SPARK_OUTPUT) else False
if spark_exists and not FORCE_RERUN:
    print("✓ Spark output exists. FORCE_RERUN=True to rebuild.")
else:
    if os.path.exists(SPARK_OUTPUT):
        shutil.rmtree(SPARK_OUTPUT)
    os.makedirs(SPARK_OUTPUT, exist_ok=True)
    os.environ.setdefault("JAVA_HOME", "/usr/lib/jvm/java-11-openjdk-amd64")
    !python scripts/run_spark_pipeline.py \
        --version {NUSCENES_VERSION} \
        --nuscenes-root {DATA_ROOT}/nuscenes \
        --output-dir {SPARK_OUTPUT} \
        2>&1 | tail -3

# ── Unified dataset ────────────────────────────────────────────────────────
UNIFIED_OUTPUT = f"{OUTPUTS_ROOT}/data/unified"
DADA_OUTPUT    = f"{OUTPUTS_ROOT}/data/dada_extracted"
ALL_MANIFEST   = f"{UNIFIED_OUTPUT}/all_manifest.jsonl"

unified_exists = os.path.exists(f"{UNIFIED_OUTPUT}/train_manifest.jsonl")
if unified_exists and not FORCE_RERUN:
    print("✓ Unified dataset exists. FORCE_RERUN=True to rebuild.")
else:
    if os.path.exists(UNIFIED_OUTPUT):
        shutil.rmtree(UNIFIED_OUTPUT)
    os.makedirs(UNIFIED_OUTPUT, exist_ok=True)
    dada_flag = (f"--dada-dir {DADA_OUTPUT}"
                 if os.path.exists(f"{DADA_OUTPUT}/metadata.jsonl") else "--nuscenes-only")
    !python scripts/run_build_unified_dataset.py \
        --nuscenes-dir {FILTER_OUTPUT} \
        --output-dir {UNIFIED_OUTPUT} \
        {dada_flag} \
        2>&1 | tail -5

# Always rebuild all_manifest.jsonl
combined = []
for split in ("train", "val", "test"):
    p = f"{UNIFIED_OUTPUT}/{split}_manifest.jsonl"
    if os.path.exists(p):
        with open(p) as f:
            for line in f:
                frame = json.loads(line)
                frame["split"] = split
                combined.append(frame)
with open(ALL_MANIFEST, "w") as f:
    for frame in combined:
        f.write(json.dumps(frame) + "\n")
print(f"✓ all_manifest.jsonl: {len(combined)} frames")

# ── Annotation ─────────────────────────────────────────────────────────────
ANNOTATION_OUTPUT = f"{OUTPUTS_ROOT}/data/annotated"
ann_exists = os.path.exists(f"{ANNOTATION_OUTPUT}/annotated_manifest.json")
if ann_exists and not FORCE_RERUN:
    print("✓ Annotation done. FORCE_RERUN=True to rerun.")
else:
    if os.path.exists(ANNOTATION_OUTPUT):
        shutil.rmtree(ANNOTATION_OUTPUT)
    os.makedirs(ANNOTATION_OUTPUT, exist_ok=True)
    if USE_MOCK_LLM:
        !python scripts/run_annotation_pipeline.py \
            --manifest {ALL_MANIFEST} \
            --output-dir {ANNOTATION_OUTPUT} \
            --mock-llm \
            2>&1 | tail -5
    else:
        try:
            from google.colab import userdata
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except Exception:
            print("⚠️ Could not load API key from Colab Secrets. Set manually:")
            print('   os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."')
            raise
        !python scripts/run_annotation_pipeline.py \
            --manifest {ALL_MANIFEST} \
            --output-dir {ANNOTATION_OUTPUT} \
            2>&1 | tail -5

# ── SFT files with correct image paths (always rebuild) ───────────────────
SFT_DIR     = f"{OUTPUTS_ROOT}/data/sft_ready"
FILTER_META = f"{FILTER_OUTPUT}/metadata.json"
IMAGES_DIR  = f"{FILTER_OUTPUT}/images"

if os.path.exists(SFT_DIR):
    shutil.rmtree(SFT_DIR)
os.makedirs(SFT_DIR, exist_ok=True)

id_to_image = {}
if os.path.exists(FILTER_META):
    with open(FILTER_META) as f:
        filtered = json.load(f)
    for frame in filtered:
        fid = frame.get("sample_token", "")
        img = frame.get("exported_image_path", "")
        if img and not os.path.isabs(img):
            img = f"{FILTER_OUTPUT}/{img}"
        if not img or not os.path.exists(img):
            cam = frame.get("cam_front_path", "")
            if cam:
                cand = f"{IMAGES_DIR}/{os.path.basename(cam)}"
                if os.path.exists(cand):
                    img = cand
        id_to_image[fid] = img

ann_path = f"{ANNOTATION_OUTPUT}/annotated_manifest.json"
if os.path.exists(ann_path):
    with open(ann_path) as f:
        annotated = json.load(f)
    SYS = ("You are DriveSense, an autonomous vehicle hazard detection system. "
           "Analyze the dashcam image and identify all safety-critical hazards. "
           "Output a structured JSON response with bounding boxes (normalized 0-1000), "
           "hazard classification, severity assessment, reasoning, and recommended action.")
    USR = ("Analyze this dashcam image for safety hazards. Identify all hazards with "
           "bounding boxes, classify each hazard, assess severity, explain your reasoning, "
           "and recommend an action. Respond with JSON only.")
    for split in ("train", "val", "test"):
        split_frames = [f for f in annotated
                        if f.get("split") == split and f.get("annotations")]
        sft_path = f"{SFT_DIR}/sft_{split}.jsonl"
        with open(sft_path, "w") as out:
            for frame in split_frames:
                fid = frame.get("frame_id", "")
                img = id_to_image.get(fid, frame.get("image_path", ""))
                out.write(json.dumps({
                    "messages": [
                        {"role": "system", "content": SYS},
                        {"role": "user", "content": [
                            {"type": "image", "image": img},
                            {"type": "text",  "text": USR},
                        ]},
                        {"role": "assistant", "content": json.dumps(frame["annotations"])},
                    ],
                    "frame_id": fid, "source": frame.get("source", ""), "split": split,
                }) + "\n")
        count = sum(1 for _ in open(sft_path))
        print(f"  ✓ sft_{split}.jsonl: {count} examples")

print("\n=== Stage 1 Complete ===")

## Stage 2: SFT Training

Checkpoints saved to Drive every 250 steps. Auto-resumes after disconnect.

In [ ]:
import os, yaml

os.chdir(REPO_ROOT)

SFT_DIR   = f"{OUTPUTS_ROOT}/data/sft_ready"
TRAIN_OUT = f"{OUTPUTS_ROOT}/training"

!cp {SFT_DIR}/sft_train.jsonl /content/sft_train.jsonl 2>/dev/null || true
!cp {SFT_DIR}/sft_val.jsonl   /content/sft_val.jsonl   2>/dev/null || true
print("✓ SFT data on fast local storage")

if RUN_DEBUG_FIRST:
    with open("configs/training.yaml") as f:
        cfg = yaml.safe_load(f)
    cfg["training"]["max_steps"]              = 10
    cfg["training"]["num_epochs"]             = 1
    cfg["training"]["logging_steps"]          = 1
    cfg["training"]["save_strategy"]          = "no"
    cfg["training"]["load_best_model_at_end"] = False
    cfg["training"]["eval_strategy"]          = "no"
    with open("configs/training_debug.yaml", "w") as f:
        yaml.dump(cfg, f)
    print("Running 10-step smoke test...")
    !python scripts/run_training.py --config configs/training_debug.yaml 2>&1 | tail -10

print("\nRunning full training...")
!python scripts/run_training.py \
    --config configs/training.yaml \
    --override training.output_dir={TRAIN_OUT} \
    --override training.save_steps=250 \
    --override training.save_total_limit=3 \
    --resume \
    2>&1
print("\n=== Stage 2 Complete ===")

## Stage 3: Model Optimization

LoRA merge → AWQ quantization → TensorRT ViT.

In [ ]:
import os, glob

os.chdir(REPO_ROOT)

TRAIN_OUT = f"{OUTPUTS_ROOT}/training"
best = f"{TRAIN_OUT}/checkpoint-best"
if not os.path.exists(best):
    ckpts = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
    best = ckpts[-1] if ckpts else None

if best and os.path.exists(best):
    !python scripts/run_optimize_model.py --all \
        --adapter-path {best} \
        --override inference.merge.output_dir={OUTPUTS_ROOT}/merged_model \
        --override inference.quantization.output_dir={OUTPUTS_ROOT}/quantized_model \
        --override inference.tensorrt.output_dir={OUTPUTS_ROOT}/tensorrt \
        2>&1 | tail -10
else:
    print("No checkpoint — running mock optimization.")
    !python scripts/run_optimize_model.py --all --mock \
        --override inference.merge.output_dir={OUTPUTS_ROOT}/merged_model \
        --override inference.quantization.output_dir={OUTPUTS_ROOT}/quantized_model \
        --override inference.tensorrt.output_dir={OUTPUTS_ROOT}/tensorrt \
        2>&1 | tail -5
print("\n=== Stage 3 Complete ===")

## Stage 4: Inference Benchmarks

In [ ]:
import os

os.chdir(REPO_ROOT)

BENCH_DIR = f"{OUTPUTS_ROOT}/benchmarks"
QUANT_DIR = f"{OUTPUTS_ROOT}/quantized_model"
os.makedirs(BENCH_DIR, exist_ok=True)
MOCK_FLAG = "" if os.path.exists(QUANT_DIR) else "--mock"

!python scripts/run_benchmark.py --vllm \
    --output {BENCH_DIR}/vllm_bench.json {MOCK_FLAG} 2>&1 | tail -5
!python scripts/run_benchmark.py --local \
    --output {BENCH_DIR}/local_bench.json {MOCK_FLAG} 2>&1 | tail -5
!python scripts/run_benchmark.py --vit-only \
    --output {BENCH_DIR}/vit_bench.json {MOCK_FLAG} 2>&1 | tail -5
print("\n=== Stage 4 Complete ===")

## Stage 5: Evaluation

In [ ]:
import os, glob

os.chdir(REPO_ROOT)

PRED_DIR  = f"{OUTPUTS_ROOT}/predictions"
PRED_FILE = f"{PRED_DIR}/test_predictions.jsonl"
EVAL_DIR  = f"{OUTPUTS_ROOT}/eval"
BENCH_DIR = f"{OUTPUTS_ROOT}/benchmarks"
TRAIN_OUT = f"{OUTPUTS_ROOT}/training"
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

best = f"{TRAIN_OUT}/checkpoint-best"
if not os.path.exists(best):
    ckpts = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
    best = ckpts[-1] if ckpts else None

adapter_flag = f"--adapter-path {best}" if best and os.path.exists(best) else "--mock"
mock_flag    = "--mock" if "--mock" in adapter_flag else ""

!python scripts/run_generate_predictions.py \
    --split test {adapter_flag} \
    --output {PRED_FILE} {mock_flag} 2>&1 | tail -5

if not USE_MOCK_JUDGE:
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        USE_MOCK_JUDGE = True
mock_judge = "--mock-judge" if USE_MOCK_JUDGE else ""

!python scripts/run_full_evaluation.py \
    --level 1 2 3 4 \
    {mock_judge} \
    --benchmarks-dir {BENCH_DIR} \
    --predictions {PRED_FILE} \
    --output-dir {EVAL_DIR} \
    {mock_flag} \
    2>&1 | tail -15

!python scripts/run_full_evaluation.py \
    --generate-report \
    --output-dir {EVAL_DIR} \
    2>&1 | tail -5

for candidate in [f"{EVAL_DIR}/final_report.txt", f"{EVAL_DIR}/evaluation_report.txt"]:
    if os.path.exists(candidate):
        print(open(candidate).read())
        break

print("\n=== Pipeline Complete ===")
print(f"All outputs at: {OUTPUTS_ROOT}")